In [6]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from scipy.sparse import hstack
import joblib

In [7]:
df = pd.read_csv('/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/CEAS_08.csv')

df.head()

,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1


In [8]:
print(df.shape)
print(df.columns)
print(df['label'].value_counts())

(39154, 7)
Index(['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls'], dtype='str')
label
1    21842
0    17312
Name: count, dtype: int64


In [9]:
# chỉ drop những dòng thiếu body hoặc label
df = df.dropna(subset=['body', 'label'])

# fill các cột khác nếu cần
df['subject'] = df['subject'].fillna('')
df['urls'] = df['urls'].fillna('')

# combine text
X_text = df['subject'] + " " + df['body']
y = df['label']

In [10]:
def extract_features(text):
    text = str(text)
    
    return [
        len(text),  # độ dài email
        len(re.findall(r'http[s]?://', text)),  # số link
        int(bool(re.search(r'urgent|verify|account|password|click', text.lower()))),  # keyword nguy hiểm
        sum(c.isdigit() for c in text),  # số chữ số
        text.count('!'),  # số dấu !
        int(bool(re.search(r'[A-Z]{5,}', text)))  # chữ in hoa bất thường
    ]

X_extra = np.array([extract_features(t) for t in X_text])

In [11]:
print(y.unique())
print(y.isna().sum())

[1 0]
0


In [12]:
X_train_text, X_test_text, X_train_extra, X_test_extra, y_train, y_test = train_test_split(
    X_text, X_extra, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1,2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

In [14]:
scaler = StandardScaler(with_mean=False)

X_train_extra_scaled = scaler.fit_transform(X_train_extra)
X_test_extra_scaled = scaler.transform(X_test_extra)

X_train_final = hstack([X_train_tfidf, X_train_extra_scaled])
X_test_final = hstack([X_test_tfidf, X_test_extra_scaled])

In [15]:
model = LinearSVC(
    C=1.5,
    class_weight='balanced',
    max_iter=5000
)

model.fit(X_train_final, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.5
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [16]:
y_pred = model.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9971906525347976

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      3462
           1       1.00      1.00      1.00      4369

    accuracy                           1.00      7831
   macro avg       1.00      1.00      1.00      7831
weighted avg       1.00      1.00      1.00      7831


Confusion Matrix:
 [[3450   12]
 [  10 4359]]


In [17]:
print("Train acc:", model.score(X_train_final, y_train))
print("Test acc:", model.score(X_test_final, y_test))

Train acc: 0.9998722983111452
Test acc: 0.9971906525347976


In [22]:
# ================== IMPORT ==================
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from scipy.sparse import hstack

# ================== LOAD DATA ==================
df = pd.read_csv('/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/phishing_email.csv')

print("Shape:", df.shape)
print(df.head())

# ================== SET COLUMN ==================
text_col = 'text_combined'
label_col = 'label'

# ================== CLEAN ==================
df = df.dropna(subset=[text_col, label_col])

X_text = df[text_col].astype(str)
y = df[label_col].astype(int)

print("\nLabel distribution:\n", y.value_counts())

# ================== FEATURE ENGINEERING ==================
def extract_features(text):
    text = str(text)
    
    return [
        len(text),
        len(re.findall(r'http[s]?://', text)),
        int(bool(re.search(r'urgent|money|bank|transfer|account|verify|login', text.lower()))),
        sum(c.isdigit() for c in text),
        text.count('!'),
        int(bool(re.search(r'[A-Z]{5,}', text)))
    ]

X_extra = np.array([extract_features(t) for t in X_text])

# ================== SPLIT ==================
X_train_text, X_test_text, X_train_extra, X_test_extra, y_train, y_test = train_test_split(
    X_text, X_extra, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ================== TF-IDF ==================
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1,2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# ================== SCALE EXTRA ==================
scaler = StandardScaler(with_mean=False)

X_train_extra_scaled = scaler.fit_transform(X_train_extra)
X_test_extra_scaled = scaler.transform(X_test_extra)

# ================== COMBINE ==================
X_train_final = hstack([X_train_tfidf, X_train_extra_scaled])
X_test_final = hstack([X_test_tfidf, X_test_extra_scaled])

# ================== TRAIN ==================
model = LinearSVC(
    C=1.5,
    class_weight='balanced',
    max_iter=5000
)

model.fit(X_train_final, y_train)

# ================== EVALUATE ==================
y_pred = model.predict(X_test_final)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ================== TEST ==================
def predict_email(text):
    text_vec = tfidf.transform([text])
    extra = np.array([extract_features(text)])
    extra_scaled = scaler.transform(extra)
    
    final = hstack([text_vec, extra_scaled])
    
    pred = model.predict(final)[0]
    return "Phishing" if pred == 1 else "Safe"

print("\nTest:")
print("1:", predict_email("Hello, how are you?"))
print("2:", predict_email("I am a prince and I want to transfer $10 million to your account"))
print("3:", predict_email("Your account has been suspended, click here to verify immediately!"))

import joblib

joblib.dump(model, 'svm_model.pkl')
joblib.dump(tfidf, 'tfidf.pkl')
joblib.dump(scaler, 'scaler.pkl')

Shape: (82486, 2)
                                       text_combined  label
0  hpl nom may 25 2001 see attached file hplno 52...      0
1  nom actual vols 24 th forwarded sabrae zajac h...      0
2  enron actuals march 30 april 1 201 estimated a...      0
3  hpl nom may 30 2001 see attached file hplno 53...      0
4  hpl nom june 1 2001 see attached file hplno 60...      0

Label distribution:
 label
1    42891
0    39595
Name: count, dtype: int64

Accuracy: 0.9880591586859013

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      7919
           1       0.99      0.99      0.99      8579

    accuracy                           0.99     16498
   macro avg       0.99      0.99      0.99     16498
weighted avg       0.99      0.99      0.99     16498


Confusion Matrix:
 [[7823   96]
 [ 101 8478]]

Test:
1: Phishing
2: Phishing
3: Phishing


['scaler.pkl']